<a href="https://colab.research.google.com/github/ifchandesh/employee_attrition_analysis/blob/main/employee_attrition_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Employee Attrition Analysis & Prediction

This notebook analyzes employee turnover using the IBM HR Analytics dataset, identifies key factors driving resignation (Age, Salary, Department, OverTime, Work-Life Balance, Distance From Home, Travel Frequency, and Tenure), builds classification models, and provides an interactive HR dashboard with KPIs.

## 1. Setup & Load Dataset

In [ ]:
# Install dependencies if running in Google Colab
!pip install -q gradio scikit-learn seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Load IBM HR Analytics dataset
data_url = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"
df = pd.read_csv(data_url)

# Drop uninformative / constant columns
drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Identify Factors Influencing Resignation (EDA)
We examine how **Age, Salary, Department, OverTime, Job Satisfaction, Work-Life Balance, Commute Distance, Business Travel, and Tenure** impact employee turnover.

In [ ]:
# Comprehensive EDA: Visualizing key factors driving attrition
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
palette = {'Yes': '#e74c3c', 'No': '#34495e'}

# 1. Age Distribution
sns.kdeplot(data=df, x='Age', hue='Attrition', fill=True, palette=palette, ax=axes[0, 0], alpha=0.5)
axes[0, 0].set_title("1. Age Distribution vs Attrition (Younger employees leave more)", fontweight='bold')
axes[0, 0].set_xlabel("Age")

# 2. Monthly Income by Department
sns.boxplot(data=df, x='Department', y='MonthlyIncome', hue='Attrition', palette=palette, ax=axes[0, 1])
axes[0, 1].set_title("2. Monthly Income & Department vs Attrition", fontweight='bold')
axes[0, 1].set_xlabel("Department")
axes[0, 1].set_ylabel("Monthly Income ($)")

# 3. OverTime Impact
ot_pct = df.groupby('OverTime')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
sns.barplot(data=ot_pct, x='OverTime', y='Attrition', hue='OverTime', palette='Reds', legend=False, ax=axes[1, 0])
axes[1, 0].set_title("3. Attrition Rate (%) by OverTime Status", fontweight='bold')
axes[1, 0].set_ylabel("Attrition Rate (%)")

# 4. Job Satisfaction Impact
sat_pct = df.groupby('JobSatisfaction')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
sns.barplot(data=sat_pct, x='JobSatisfaction', y='Attrition', hue='JobSatisfaction', palette='Blues_r', legend=False, ax=axes[1, 1])
axes[1, 1].set_title("4. Attrition Rate (%) by Job Satisfaction (1=Low, 4=High)", fontweight='bold')
axes[1, 1].set_ylabel("Attrition Rate (%)")
axes[1, 1].set_xlabel("Job Satisfaction Rating")

# 5. Work-Life Balance Impact
wlb_pct = df.groupby('WorkLifeBalance')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
sns.barplot(data=wlb_pct, x='WorkLifeBalance', y='Attrition', hue='WorkLifeBalance', palette='Purples_r', legend=False, ax=axes[2, 0])
axes[2, 0].set_title("5. Attrition Rate (%) by Work-Life Balance (1=Bad, 4=Best)", fontweight='bold')
axes[2, 0].set_ylabel("Attrition Rate (%)")
axes[2, 0].set_xlabel("Work-Life Balance Rating")

# 6. Distance From Home Impact
sns.boxplot(data=df, x='Attrition', y='DistanceFromHome', hue='Attrition', palette=palette, legend=False, ax=axes[2, 1])
axes[2, 1].set_title("6. Distance From Home (Miles) vs Attrition", fontweight='bold')
axes[2, 1].set_xlabel("Attrition")
axes[2, 1].set_ylabel("Distance From Home (Miles)")

# 7. Business Travel Frequency
travel_pct = df.groupby('BusinessTravel')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
sns.barplot(data=travel_pct, x='BusinessTravel', y='Attrition', hue='BusinessTravel', palette='Oranges_r', legend=False, ax=axes[3, 0])
axes[3, 0].set_title("7. Attrition Rate (%) by Business Travel Frequency", fontweight='bold')
axes[3, 0].set_ylabel("Attrition Rate (%)")
axes[3, 0].set_xlabel("Travel Frequency")

# 8. Tenure Distribution (Years at Company)
sns.kdeplot(data=df, x='YearsAtCompany', hue='Attrition', fill=True, palette=palette, ax=axes[3, 1], alpha=0.5)
axes[3, 1].set_title("8. Tenure (Years at Company) vs Attrition", fontweight='bold')
axes[3, 1].set_xlabel("Years at Company")

plt.tight_layout()
plt.show()

## 3. Classification Algorithms & Model Evaluation
We prepare features, train **Logistic Regression, Decision Tree, and Random Forest** models with 5-fold cross validation, evaluate metrics, and visualize confusion matrices.

In [ ]:
# Prepare features and target
X = df.drop(columns=['Attrition'])
y = (df['Attrition'] == 'Yes').astype(int)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train and evaluate models
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42)
}

results = []
trained_pipes = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    # 5-fold cross-validated ROC-AUC score
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()

    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall (Sensitivity)': round(recall_score(y_test, y_pred), 3),
        'F1-Score': round(f1_score(y_test, y_pred), 3),
        'Test ROC-AUC': round(roc_auc_score(y_test, y_proba), 3),
        '5-Fold CV ROC-AUC': round(cv_auc, 3)
    })
    trained_pipes[name] = pipe

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Visualizing Confusion Matrices across all models
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, (name, pipe) in enumerate(trained_pipes.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Resigned'])
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontweight='bold')
    axes[idx].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Top Feature Importances from Random Forest
rf_model = trained_pipes['Random Forest'].named_steps['classifier']
encoded_cat_names = trained_pipes['Random Forest'].named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_features = num_cols + list(encoded_cat_names)

feat_imp = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 4))
feat_imp.sort_values().plot(kind='barh', color='#2980b9')
plt.title("Top 10 Most Important Features in Predicting Attrition", fontweight='bold')
plt.xlabel("Importance")
plt.show()

## 4. HR KPI Dashboard & Flight Risk Predictor
An interactive dashboard displaying key HR metrics and allowing real-time prediction of employee flight risk based on comprehensive employee profile attributes.

In [ ]:
# Compute HR KPIs
total_headcount = len(df)
total_attrition = (df['Attrition'] == 'Yes').sum()
attrition_rate = round((total_attrition / total_headcount) * 100, 1)
avg_income = round(df['MonthlyIncome'].mean(), 2)

best_pipe = trained_pipes['Random Forest']

def predict_attrition(age, department, job_role, monthly_income, overtime, job_satisfaction, work_life_balance, distance_from_home, business_travel, stock_option_level, years_at_company):
    sample = pd.DataFrame([{
        'Age': age,
        'BusinessTravel': business_travel,
        'DailyRate': 800,
        'Department': department,
        'DistanceFromHome': distance_from_home,
        'Education': 3,
        'EducationField': 'Life Sciences',
        'EnvironmentSatisfaction': 3,
        'Gender': 'Male',
        'HourlyRate': 65,
        'JobInvolvement': 3,
        'JobLevel': 2,
        'JobRole': job_role,
        'JobSatisfaction': job_satisfaction,
        'MaritalStatus': 'Single',
        'MonthlyIncome': monthly_income,
        'MonthlyRate': 14000,
        'NumCompaniesWorked': 2,
        'OverTime': overtime,
        'PercentSalaryHike': 15,
        'PerformanceRating': 3,
        'RelationshipSatisfaction': 3,
        'StockOptionLevel': stock_option_level,
        'TotalWorkingYears': max(years_at_company, max(0, age - 18)),
        'TrainingTimesLastYear': 3,
        'WorkLifeBalance': work_life_balance,
        'YearsAtCompany': years_at_company,
        'YearsInCurrentRole': max(0, years_at_company - 1),
        'YearsSinceLastPromotion': 1,
        'YearsWithCurrManager': max(0, years_at_company - 1)
    }])

    prob = best_pipe.predict_proba(sample)[0][1]
    prob_pct = round(prob * 100, 1)
    status = "High Risk (Likely to leave)" if prob_pct >= 50 else ("Moderate Risk" if prob_pct >= 25 else "Low Risk (Likely to stay)")
    return f"{prob_pct}%", status

# Create Gradio Dashboard
job_roles = [
    'Healthcare Representative',
    'Human Resources',
    'Laboratory Technician',
    'Manager',
    'Manufacturing Director',
    'Research Director',
    'Research Scientist',
    'Sales Executive',
    'Sales Representative'
]

with gr.Blocks(title="HR Attrition Dashboard") as demo:
    gr.Markdown("# HR Employee Attrition Dashboard")

    # KPI Cards
    with gr.Row():
        gr.Textbox(value=f"{total_headcount:,}", label="Total Employees", interactive=False)
        gr.Textbox(value=f"{attrition_rate}% ({total_attrition} resigned)", label="Overall Attrition Rate", interactive=False)
        gr.Textbox(value=f"${avg_income:,.2f}", label="Average Monthly Income", interactive=False)

    # Predictor Controls
    gr.Markdown("### Employee Flight Risk Predictor")
    with gr.Row():
        with gr.Column():
            gr.Markdown("**Role & Compensation**")
            age_in = gr.Slider(18, 65, value=30, step=1, label="Age")
            dept_in = gr.Dropdown(['Sales', 'Research & Development', 'Human Resources'], value='Sales', label="Department")
            role_in = gr.Dropdown(job_roles, value='Sales Representative', label="Job Role")
            income_in = gr.Slider(1000, 20000, value=3500, step=100, label="Monthly Income ($)")
        with gr.Column():
            gr.Markdown("**Work Environment & Well-being**")
            ot_in = gr.Radio(['Yes', 'No'], value='Yes', label="OverTime")
            sat_in = gr.Slider(1, 4, value=2, step=1, label="Job Satisfaction (1=Low, 4=High)")
            wlb_in = gr.Slider(1, 4, value=2, step=1, label="Work-Life Balance (1=Bad, 4=Best)")
            travel_in = gr.Dropdown(['Non-Travel', 'Travel_Rarely', 'Travel_Frequently'], value='Travel_Rarely', label="Business Travel")
        with gr.Column():
            gr.Markdown("**Tenure & Mobility**")
            dist_in = gr.Slider(1, 30, value=10, step=1, label="Distance From Home (Miles)")
            stock_in = gr.Slider(0, 3, value=0, step=1, label="Stock Option Level (0=None, 3=Max)")
            tenure_in = gr.Slider(0, 40, value=2, step=1, label="Years at Company")

    predict_btn = gr.Button("Predict Flight Risk", variant="primary")

    with gr.Row():
        out_prob = gr.Textbox(label="Attrition Probability")
        out_status = gr.Textbox(label="Risk Level")

    predict_btn.click(
        predict_attrition,
        inputs=[age_in, dept_in, role_in, income_in, ot_in, sat_in, wlb_in, dist_in, travel_in, stock_in, tenure_in],
        outputs=[out_prob, out_status]
    )

# Launch Dashboard in Colab
demo.launch(share=True)